# 🖼️ Image Processing Utility Tool
**Author:** Shreya Mishra | BIT Mesra, Jaipur  
**Date:** Oct 2024 – Dec 2024  
**Tools:** Pillow (PIL), NumPy, Matplotlib

---
### Objective
Build a batch image processing pipeline that applies grayscale conversion, resizing, brightness adjustment, blur filtering, and edge detection across multiple images. Visualise pixel intensity histograms for each image.


In [ ]:
# Install dependencies (only needed in Colab)
# !pip install Pillow numpy matplotlib

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageEnhance
from pathlib import Path

print('Libraries imported ✅')

## 1. Configuration

In [ ]:
INPUT_DIR  = 'sample_images'
OUTPUT_DIR = 'processed_images'
RESIZE_TO  = (256, 256)
BRIGHTNESS = 1.3

os.makedirs(INPUT_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Folders ready ✅')

## 2. Generate Demo Images
*(Skip this cell if you have your own images in `sample_images/`)*

In [ ]:
colors = [(200,80,80),(80,160,200),(120,200,90),(230,180,60),(160,90,210)]
for i, c in enumerate(colors):
    arr  = np.full((300,300,3), c, dtype=np.uint8)
    arr += np.random.randint(0, 40, arr.shape, dtype=np.uint8)
    Image.fromarray(np.clip(arr,0,255).astype(np.uint8)).save(f'{INPUT_DIR}/demo_{i+1}.png')
print('5 demo images created in sample_images/ ✅')

## 3. Processing Pipeline Function

In [ ]:
def process_image(path):
    img        = Image.open(path).convert('RGB')
    resized    = img.resize(RESIZE_TO, Image.LANCZOS)
    grayscale  = resized.convert('L')
    brightened = ImageEnhance.Brightness(resized).enhance(BRIGHTNESS)
    blurred    = resized.filter(ImageFilter.GaussianBlur(radius=2))
    edges      = resized.filter(ImageFilter.FIND_EDGES).convert('L')
    arr        = np.array(grayscale)
    return {
        'name'      : Path(path).stem,
        'original'  : resized,
        'grayscale' : grayscale,
        'brightened': brightened,
        'blurred'   : blurred,
        'edges'     : edges,
        'arr'       : arr,
        'stats'     : {'mean': arr.mean().round(2), 'std': arr.std().round(2),
                       'min': int(arr.min()), 'max': int(arr.max())}
    }

print('Pipeline function defined ✅')

## 4. Batch Process All Images

In [ ]:
paths   = list(Path(INPUT_DIR).glob('*.png')) + list(Path(INPUT_DIR).glob('*.jpg'))
results = []

for p in paths:
    r = process_image(str(p))
    results.append(r)
    r['grayscale'].save(f"{OUTPUT_DIR}/{r['name']}_gray.png")
    r['brightened'].save(f"{OUTPUT_DIR}/{r['name']}_bright.png")
    r['blurred'].save(f"{OUTPUT_DIR}/{r['name']}_blur.png")
    r['edges'].save(f"{OUTPUT_DIR}/{r['name']}_edges.png")
    s = r['stats']
    print(f"✓ {r['name']:15s}  mean={s['mean']}  std={s['std']}  min={s['min']}  max={s['max']}")

print(f'\nAll outputs saved to {OUTPUT_DIR}/ ✅')

## 5. Visualise Pipeline – First Image

In [ ]:
r0     = results[0]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle(f"Pipeline Demo – {r0['name']}", fontsize=14, fontweight='bold')

panels = [
    (r0['original'],   'Original (resized)', None),
    (r0['grayscale'],  'Grayscale',          'gray'),
    (r0['brightened'], f'Brightness x{BRIGHTNESS}', None),
    (r0['blurred'],    'Gaussian Blur',      None),
    (r0['edges'],      'Edge Detection',     'gray'),
]
for ax, (im, title, cmap) in zip(axes.flat, panels):
    ax.imshow(im, cmap=cmap)
    ax.set_title(title)
    ax.axis('off')

axes[1,2].hist(r0['arr'].ravel(), bins=50, color='steelblue', edgecolor='white')
axes[1,2].set_title('Pixel Intensity Histogram')
axes[1,2].set_xlabel('Pixel Value (0–255)')
axes[1,2].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('image_processing_demo.png', dpi=150)
plt.show()

## 6. Batch Pixel Histogram – All Images

In [ ]:
fig2, axes2 = plt.subplots(1, len(results), figsize=(4*len(results), 4))
if len(results) == 1:
    axes2 = [axes2]
for ax, r in zip(axes2, results):
    ax.hist(r['arr'].ravel(), bins=40, color='coral', edgecolor='white')
    ax.set_title(r['name'])
    ax.set_xlabel('Pixel Value')
fig2.suptitle('Pixel Histograms – All Images', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('batch_histograms.png', dpi=150)
plt.show()
print('Saved → batch_histograms.png ✅')